# Garfield NSCLC Train/Test Query-to-Reference Tutorial

This notebook is a compact, reviewer-facing train/test version of the NSCLC spatial query-to-reference workflow used in `Garfield_code/spatial-single-modal-code/spatial_niche_NSCLC.ipynb`.

It holds out the `lung13` batch as query data, trains the spatial reference model on the remaining NSCLC batches, maps the held-out query back to the reference with Garfield's `load_query_data` path, and evaluates label transfer after training. Cell-type and niche labels are removed from the AnnData objects passed into Garfield and are reattached only for post-hoc evaluation and label transfer.

## 1. Setup

In [ ]:
import os
import sys
import warnings
from pathlib import Path

warnings.simplefilter("ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")
os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba-cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl-cache")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/xdg-cache")
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("MKL_NUM_THREADS", "8")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "8")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "8")

REPO = Path.cwd().resolve()
if not (REPO / "Garfield-garfield_dev").exists():
    REPO = Path("/data2/zhouwg_data/project/Garfield-reproducibility").resolve()
sys.path.insert(0, str(REPO / "Garfield-garfield_dev"))

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import Garfield as gf
from Garfield.model import Garfield
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    balanced_accuracy_score,
    f1_score,
    normalized_mutual_info_score,
)

print(f"Repository: {REPO}")
print(f"Garfield import path: {Path(gf.__file__).resolve()}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
DATA_DIR = REPO / "datasets" / "st_data" / "gold"
WORKDIR = REPO / "revision_experiments" / "_runs" / "tutorial_train_test_nsclc_spatial"
RESULTS_DIR = REPO / "revision_experiments" / "results" / "tutorial_train_test_nsclc_spatial"
MODEL_DIR = WORKDIR / "model_ref"
TRANSFER_MODEL_DIR = WORKDIR / "model_transfer"

BATCH_KEY = "batch"
QUERY_BATCH = "lung13"
CELLTYPE_KEY = "cell_type"
MINOR_CELLTYPE_KEY = "cell_type_original"
BROAD_NICHE_KEY = "niche"
TRANSFER_NICHE_KEY = "niche_type"
LATENT_KEY = "garfield_latent"
SOURCE_NOTEBOOK = REPO / "Garfield_code" / "spatial-single-modal-code" / "spatial_niche_NSCLC.ipynb"

# Default: an easy-to-run smoke test on the 1% NSCLC files. For a paper-scale
# rerun, set USE_FULL_DATA=True, both MAX_* values to None, and increase epochs.
USE_FULL_DATA = False
MAX_REF_CELLS_PER_BATCH = 500
MAX_QUERY_CELLS = 500
REFERENCE_EPOCHS = 3
QUERY_EPOCHS = 2

RUN_REFERENCE_TRAINING = True
RUN_QUERY_SURGERY = True
SAVE_OUTPUTS = True

SEED = 2024
DEVICE_ID = 0
rng = np.random.default_rng(SEED)

DATA_PATTERN = (
    "nanostring_cosmx_human_nsclc_batch*.h5ad"
    if USE_FULL_DATA
    else "nanostring_cosmx_human_nsclc_subsample_1pct_batch*.h5ad"
)
print(f"Using data pattern: {DATA_PATTERN}")

## 2. Relation to the paper workflow

The original paper notebook trains a spatial single-modal NSCLC reference on all batches except `lung13` and then maps `lung13` as query. The key settings reused here are `profile='spatial'`, `data_type='single-modal'`, `graph_const_method='Squidpy'`, `weight=0.5`, `used_mmd=True`, and query fine-tuning with `sample_col='projection'`.

This tutorial reduces cells and epochs by default. The workflow and held-out split match the paper notebook; the default numbers are for demonstrating reproducibility mechanics, not for replacing the full paper-scale result.

## 3. Load NSCLC spatial data and hold out `lung13`

In [ ]:
def sample_adata(adata, max_cells, rng):
    if max_cells is None or adata.n_obs <= max_cells:
        return adata.copy()
    idx = rng.choice(adata.n_obs, size=max_cells, replace=False)
    return adata[idx].copy()


batch_adatas = []
load_rows = []
for path in sorted(DATA_DIR.glob(DATA_PATTERN)):
    a = sc.read_h5ad(path)
    assert BATCH_KEY in a.obs, f"{path.name} lacks obs[{BATCH_KEY!r}]"
    assert "spatial" in a.obsm, f"{path.name} lacks obsm['spatial']"
    if "counts" in a.layers:
        a.X = a.layers["counts"].copy()
    a.var_names_make_unique()

    batch_name = str(a.obs[BATCH_KEY].astype(str).iloc[0])
    max_cells = MAX_QUERY_CELLS if batch_name == QUERY_BATCH else MAX_REF_CELLS_PER_BATCH
    original_n = int(a.n_obs)
    a = sample_adata(a, max_cells=max_cells, rng=rng)
    batch_adatas.append(a)
    load_rows.append({
        "file": path.name,
        BATCH_KEY: batch_name,
        "original_n_obs": original_n,
        "used_n_obs": int(a.n_obs),
        "split": "query" if batch_name == QUERY_BATCH else "reference",
    })

assert batch_adatas, f"No files matched {DATA_PATTERN} under {DATA_DIR}"
adata_all = batch_adatas[0].concatenate(batch_adatas[1:], batch_key="Batch")
adata_all.obs_names_make_unique()
if "counts" in adata_all.layers:
    adata_all.X = adata_all.layers["counts"].copy()

split_summary = pd.DataFrame(load_rows)
display(split_summary)
print(adata_all)
print(adata_all.obs[BATCH_KEY].value_counts())

In [ ]:
label_columns = [BATCH_KEY, CELLTYPE_KEY, MINOR_CELLTYPE_KEY, BROAD_NICHE_KEY]
missing_label_cols = [c for c in label_columns if c not in adata_all.obs]
assert not missing_label_cols, f"Missing expected label columns: {missing_label_cols}"
labels_by_obs = adata_all.obs[label_columns].copy()

reference_raw = adata_all[~adata_all.obs[BATCH_KEY].astype(str).isin([QUERY_BATCH]), :].copy()
query_raw = adata_all[adata_all.obs[BATCH_KEY].astype(str).isin([QUERY_BATCH]), :].copy()
assert reference_raw.n_obs > 0 and query_raw.n_obs > 0

def strip_supervised_labels(adata):
    stripped = adata.copy()
    keep = [c for c in [BATCH_KEY, "Batch"] if c in stripped.obs]
    stripped.obs = stripped.obs[keep].copy()
    return stripped


reference_input = strip_supervised_labels(reference_raw)
query_input = strip_supervised_labels(query_raw)

assert CELLTYPE_KEY not in reference_input.obs
assert MINOR_CELLTYPE_KEY not in reference_input.obs
assert BROAD_NICHE_KEY not in reference_input.obs
assert CELLTYPE_KEY not in query_input.obs
assert BROAD_NICHE_KEY not in query_input.obs

print(f"Reference cells: {reference_input.n_obs}")
print(f"Query cells: {query_input.n_obs}")
print("Labels stripped from Garfield inputs: True")

## 4. Configure the spatial reference model

In [ ]:
gf.settings.set_workdir(str(WORKDIR))
gf.settings.set_gf_params({})

node_batch_size = 512 if USE_FULL_DATA else 128
spatial_reference_overrides = dict(
    adata_list=reference_input,
    profile="spatial",
    data_type="single-modal",
    sub_data_type=None,
    sample_col=None,
    weight=0.5,
    graph_const_method="Squidpy",
    used_hvg=True,
    min_cells=3,
    min_features=0,
    keep_mt=False,
    target_sum=1e4,
    rna_n_top_features=960,
    n_components=50,
    n_neighbors=5,
    metric="euclidean",
    svd_solver="arpack",
    used_pca_feat=False,
    adj_key="connectivities",
    edge_val_ratio=0.1,
    edge_test_ratio=0.0,
    node_val_ratio=0.1,
    node_test_ratio=0.0,
    augment_type="svd",
    svd_q=5,
    use_FCencoder=True,
    conv_type="GAT",
    gnn_layer=2,
    hidden_dims=[128, 128],
    bottle_neck_neurons=20,
    cluster_num=20,
    drop_feature_rate=0.2,
    drop_edge_rate=0.2,
    num_heads=3,
    dropout=0.2,
    concat=True,
    used_edge_weight=True,
    used_DSBN=False,
    used_mmd=True,
    num_neighbors=5,
    loaders_n_hops=2,
    edge_batch_size=4096,
    node_batch_size=node_batch_size,
    include_edge_recon_loss=True,
    include_gene_expr_recon_loss=True,
    lambda_latent_contrastive_instanceloss=1.0,
    lambda_latent_contrastive_clusterloss=0.5,
    lambda_gene_expr_recon=1.0,
    lambda_edge_recon=10.0,
    lambda_latent_adj_recon_loss=2.0,
    lambda_omics_recon_mmd_loss=0.5,
    n_epochs_no_edge_recon=0,
    n_epochs=REFERENCE_EPOCHS,
    learning_rate=0.001,
    weight_decay=1e-5,
    gradient_clipping=5,
    latent_key=LATENT_KEY,
    reload_best_model=True,
    use_early_stopping=True,
    early_stopping_kwargs={"patience": max(4, REFERENCE_EPOCHS), "lr_patience": 2},
    monitor=True,
    device_id=DEVICE_ID,
    seed=SEED,
    verbose=True,
    user_cache_path=str(WORKDIR / "cache_ref"),
    use_lightning=False,
)

train_config = dict(gf.settings.gf_params)
train_config.update(spatial_reference_overrides)

assert train_config["sample_col"] is None
assert CELLTYPE_KEY not in train_config
assert BROAD_NICHE_KEY not in train_config
pd.Series({
    "profile": train_config["profile"],
    "data_type": train_config["data_type"],
    "graph_const_method": train_config["graph_const_method"],
    "used_mmd": train_config["used_mmd"],
    "reference_sample_col": train_config["sample_col"],
    "reference_epochs": train_config["n_epochs"],
    "node_batch_size": train_config["node_batch_size"],
}).to_frame("value")

## 5. Evaluation helpers

In [ ]:
NICHE_TYPE_MAP = {
    "0": "Lung9_tumor",
    "1": "Lung6_tumor",
    "2": "Neu_fibro_mixing",
    "3": "Imm_cell_enriched",
    "4": "Fibro_plasma_mixing",
    "5": "Lung5_tumor",
    "6": "Neutrophil_expansion",
    "7": "Lymphoid_aggregates",
    "8": "Myeloid_enriched",
    "9": "Memory_CD8T_tumor",
    "10": "Fibro_Treg_mixing",
    "11": "Neu_infiltrated_tumor",
    "12": "Endo_enriched_tumor",
    "13": "Lung12_tumor",
    "14": "Epithelial_enriched",
    "15": "Mast_enriched",
}


def attach_original_labels(adata, label_table=labels_by_obs):
    aligned = label_table.reindex(adata.obs_names)
    missing = int(aligned[CELLTYPE_KEY].isna().sum())
    if missing:
        raise ValueError(f"Could not align original labels for {missing} observations")
    for col in label_table.columns:
        adata.obs[col] = aligned[col].astype(str).values
    return adata


def add_reference_niche_type(adata, cluster_key):
    clusters = adata.obs[cluster_key].astype(str)
    mapped = clusters.map(NICHE_TYPE_MAP)
    adata.obs[TRANSFER_NICHE_KEY] = mapped.fillna("cluster_" + clusters).astype("category")
    return adata


def clustering_metrics(adata, cluster_key, label_key):
    obs = adata.obs[[cluster_key, label_key]].dropna()
    return {
        "label_key": label_key,
        "n_obs": int(obs.shape[0]),
        "n_clusters": int(obs[cluster_key].nunique()),
        "n_labels": int(obs[label_key].nunique()),
        "ARI": float(adjusted_rand_score(obs[label_key].astype(str), obs[cluster_key].astype(str))),
        "NMI": float(normalized_mutual_info_score(obs[label_key].astype(str), obs[cluster_key].astype(str))),
    }


def score_transfer(query_obs, label_key=CELLTYPE_KEY):
    pred_key = f"transferred_{label_key}_unfiltered"
    mask = query_obs[label_key].notna() & query_obs[pred_key].notna()
    y_true = query_obs.loc[mask, label_key].astype(str)
    y_pred = query_obs.loc[mask, pred_key].astype(str)
    return {
        "label_key": label_key,
        "prediction_key": pred_key,
        "n_query_scored": int(mask.sum()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    }

## 6. Train the spatial reference model

In [ ]:
if RUN_REFERENCE_TRAINING:
    model = Garfield(train_config)
    model.train()
else:
    model = Garfield.load(dir_path=str(MODEL_DIR), adata_file_name="adata_nsclc.h5ad")

sc.pp.neighbors(model.adata, use_rep=LATENT_KEY, key_added=LATENT_KEY)
sc.tl.umap(model.adata, neighbors_key=LATENT_KEY)
latent_leiden_resolution = 0.5
latent_cluster_key = f"latent_leiden_{latent_leiden_resolution}"
sc.tl.leiden(
    adata=model.adata,
    resolution=latent_leiden_resolution,
    key_added=latent_cluster_key,
    neighbors_key=LATENT_KEY,
)

attach_original_labels(model.adata)
add_reference_niche_type(model.adata, latent_cluster_key)

reference_metrics = pd.DataFrame([
    clustering_metrics(model.adata, latent_cluster_key, CELLTYPE_KEY),
    clustering_metrics(model.adata, latent_cluster_key, BROAD_NICHE_KEY),
])
display(reference_metrics)
display(model.adata.obs[[BATCH_KEY, CELLTYPE_KEY, BROAD_NICHE_KEY, latent_cluster_key, TRANSFER_NICHE_KEY]].head())

In [ ]:
if SAVE_OUTPUTS or RUN_QUERY_SURGERY:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    eval_obs = model.adata.obs.copy()
    supervised_cols = [CELLTYPE_KEY, MINOR_CELLTYPE_KEY, BROAD_NICHE_KEY, TRANSFER_NICHE_KEY]
    model.adata.obs = model.adata.obs.drop(columns=supervised_cols, errors="ignore")
    try:
        model.save(
            dir_path=str(MODEL_DIR),
            overwrite=True,
            save_adata=True,
            adata_file_name="adata_nsclc.h5ad",
        )
    finally:
        model.adata.obs = eval_obs
    print(f"Saved label-stripped reference model to: {MODEL_DIR}")

## 7. Map the held-out `lung13` query batch

In [ ]:
if RUN_QUERY_SURGERY:
    query_overrides = dict(
        n_epochs=QUERY_EPOCHS,
        sample_col="projection",
        used_mmd=True,
        lambda_omics_recon_mmd_loss=1.0,
        user_cache_path=str(WORKDIR / "cache_query"),
        seed=SEED,
        device_id=DEVICE_ID,
        accelerator="auto",
        devices=1,
        num_nodes=1,
        strategy="auto",
        precision="32",
        num_workers=4,
        persistent_workers=False,
        accumulate_grad_batches=1,
        logger="tensorboard",
        log_every_n_steps=50,
        log_style="auto",
        fast_dev_run=False,
        limit_train_batches=1.0,
        limit_val_batches=1.0,
        checkpoint_dir=None,
        save_top_k=1,
        save_last=True,
        lightning_sampling_mode="auto",
        use_lightning=False,
        monitor=True,
        verbose=True,
        reload_best_model=True,
        early_stopping_kwargs={"patience": max(4, QUERY_EPOCHS), "lr_patience": 2},
    )
    new_model = Garfield.load_query_data(
        dir_path=str(MODEL_DIR),
        query_adata=query_input,
        ref_adata_name="adata_nsclc.h5ad",
        batch_key=BATCH_KEY,
        use_cuda=torch.cuda.is_available(),
        unfreeze_all_weights=False,
        unfreeze_eps_weight=True,
        unfreeze_layer0=True,
        **query_overrides,
    )
    new_model.train()
else:
    new_model = Garfield.load(dir_path=str(TRANSFER_MODEL_DIR), adata_file_name="adata_concat.h5ad")

adata_ref = new_model.adata[new_model.adata.obs["projection"] == "reference", :].copy()
adata_query = new_model.adata[new_model.adata.obs["projection"] == "query", :].copy()
attach_original_labels(adata_ref)
add_reference_niche_type(adata_ref, latent_cluster_key)
attach_original_labels(adata_query)
print(f"Mapped reference cells: {adata_ref.n_obs}")
print(f"Mapped query cells: {adata_query.n_obs}")

## 8. Label transfer and held-out evaluation

In [ ]:
n_transfer_neighbors = min(10, max(1, adata_ref.n_obs - 1))

adata_query = new_model.label_transfer(
    ref_adata=adata_ref,
    ref_adata_emb=LATENT_KEY,
    query_adata=adata_query,
    query_adata_emb=LATENT_KEY,
    n_neighbors=n_transfer_neighbors,
    ref_adata_obs=adata_ref.obs,
    label_keys=TRANSFER_NICHE_KEY,
)

# This call transfers both cell_type and cell_type_original because Garfield
# selects obs columns that start with the provided label prefix.
adata_query = new_model.label_transfer(
    ref_adata=adata_ref,
    ref_adata_emb=LATENT_KEY,
    query_adata=adata_query,
    query_adata_emb=LATENT_KEY,
    n_neighbors=n_transfer_neighbors,
    ref_adata_obs=adata_ref.obs,
    label_keys=CELLTYPE_KEY,
)

celltype_transfer_metrics = pd.DataFrame([score_transfer(adata_query.obs, CELLTYPE_KEY)])
display(celltype_transfer_metrics)
display(pd.crosstab(
    adata_query.obs[CELLTYPE_KEY],
    adata_query.obs[f"transferred_{CELLTYPE_KEY}_unfiltered"],
    normalize="index",
).round(3))

query_transfer_summary = pd.DataFrame({
    "transferred_niche_type_counts": adata_query.obs[f"transferred_{TRANSFER_NICHE_KEY}_unfiltered"].value_counts(),
})
display(query_transfer_summary)

## 9. Leakage audit and outputs

In [ ]:
audit = {
    "source_notebook": str(SOURCE_NOTEBOOK.relative_to(REPO)),
    "data_pattern": DATA_PATTERN,
    "use_full_data": USE_FULL_DATA,
    "query_batch": QUERY_BATCH,
    "reference_batches": ", ".join(split_summary.loc[split_summary.split == "reference", BATCH_KEY].astype(str)),
    "reference_sample_col": train_config["sample_col"],
    "query_sample_col": "projection",
    "celltype_key_in_config": CELLTYPE_KEY in train_config,
    "broad_niche_key_in_config": BROAD_NICHE_KEY in train_config,
    "labels_in_reference_input": any(c in reference_input.obs for c in [CELLTYPE_KEY, MINOR_CELLTYPE_KEY, BROAD_NICHE_KEY]),
    "labels_in_query_input": any(c in query_input.obs for c in [CELLTYPE_KEY, MINOR_CELLTYPE_KEY, BROAD_NICHE_KEY]),
    "reference_epochs": REFERENCE_EPOCHS,
    "query_epochs": QUERY_EPOCHS,
}

assert audit["celltype_key_in_config"] is False
assert audit["broad_niche_key_in_config"] is False
assert audit["labels_in_reference_input"] is False
assert audit["labels_in_query_input"] is False
assert QUERY_BATCH not in split_summary.loc[split_summary.split == "reference", BATCH_KEY].astype(str).tolist()

audit_df = pd.Series(audit).to_frame("value")
display(audit_df)

if SAVE_OUTPUTS:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    split_summary.to_csv(RESULTS_DIR / "split_summary.csv", index=False)
    audit_df.to_csv(RESULTS_DIR / "leakage_audit.csv")
    reference_metrics.to_csv(RESULTS_DIR / "reference_posthoc_metrics.csv", index=False)
    celltype_transfer_metrics.to_csv(RESULTS_DIR / "query_celltype_transfer_metrics.csv", index=False)
    query_transfer_summary.to_csv(RESULTS_DIR / "query_transfer_niche_counts.csv")
    query_to_save = adata_query.copy()
    for col in query_to_save.obs.columns:
        if query_to_save.obs[col].dtype == object:
            numeric_col = pd.to_numeric(query_to_save.obs[col], errors="coerce")
            if numeric_col.notna().sum() == query_to_save.obs[col].notna().sum():
                query_to_save.obs[col] = numeric_col
            else:
                query_to_save.obs[col] = query_to_save.obs[col].astype(str)
    query_to_save.write_h5ad(RESULTS_DIR / "heldout_lung13_query_with_predictions.h5ad")
    if RUN_QUERY_SURGERY:
        TRANSFER_MODEL_DIR.mkdir(parents=True, exist_ok=True)
        new_model.save(
            dir_path=str(TRANSFER_MODEL_DIR),
            overwrite=True,
            save_adata=True,
            adata_file_name="adata_concat.h5ad",
        )
    print(f"Saved tutorial outputs to: {RESULTS_DIR}")